In [ ]:
# oh boy. We'll build this off the VLA trainer, though that's a massive mess, but WANDB support is nice.
# If you're reading this, take a break. Go outside and get some fresh air.
# Dealing with this script will make you want to kill yourself.

In [ ]:

# TODO: NOT ON THE NOTEBOOK DAMN
def _prepare_feature_sequences(feature_sequences, window_size, name):
    if isinstance(feature_sequences, (np.ndarray, torch.Tensor)):
        if feature_sequences.ndim == 3:
            sequences = [feature_sequences]
        elif feature_sequences.ndim == 4:
            sequences = list(feature_sequences)
        else:
            raise ValueError(f"{name} must contain sequences shaped (F, N, 13)")
    else:
        sequences = list(feature_sequences)

    if not sequences:
        raise ValueError(f"{name} must contain at least one sequence")

    prepared_sequences = []
    for sequence_index, sequence in enumerate(sequences):
        sequence = torch.as_tensor(sequence, dtype=torch.float32).detach().cpu()
        if sequence.ndim != 3 or sequence.shape[-1] != 13:
            raise ValueError(f"{name}[{sequence_index}] must have shape (F, N, 13)")
        if sequence.shape[0] <= window_size:
            raise ValueError(
                f"{name}[{sequence_index}] needs more frames than window_size"
            )
        if sequence.shape[1] == 0:
            raise ValueError(f"{name}[{sequence_index}] must contain entities")
        if not torch.isfinite(sequence).all():
            raise ValueError(f"{name}[{sequence_index}] must contain finite values")

        track_ids = sequence[:, :, 0]
        if not torch.equal(track_ids, track_ids[0:1, :].expand_as(track_ids)):
            raise ValueError(
                f"{name}[{sequence_index}] must keep each track in the same row"
            )
        if any(torch.unique(frame_ids).numel() != sequence.shape[1] for frame_ids in track_ids):
            raise ValueError(
                f"{name}[{sequence_index}] needs unique track IDs in every frame"
            )
        prepared_sequences.append(sequence)

    return prepared_sequences

def train_stgnn(
    model,
    training_sequences,
    window_size=8,
    epochs=20,
    learning_rate=1e-3,
    device=None,
):
    """Train one-step motion forecasts from tracked feature sequences.

    Each sequence has shape (F, N, 13). Every sliding input window predicts
    the following frame's [px, py, vx, vy] values using smooth L1 loss.
    Returns the mean training loss for each epoch.
    """
    if window_size <= 0:
        raise ValueError("window_size must be positive")
    if epochs <= 0:
        raise ValueError("epochs must be positive")
    if learning_rate <= 0:
        raise ValueError("learning_rate must be positive")

    sequences = _prepare_feature_sequences(
        training_sequences,
        window_size,
        "training_sequences",
    )
    training_windows = [
        (sequence, start)
        for sequence in sequences
        for start in range(sequence.shape[0] - window_size)
    ]

    if device is not None:
        model.to(device)
    model_device = next(model.parameters()).device
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    epoch_losses = []

    for _ in range(epochs):
        model.train()
        total_loss = 0.0
        window_order = torch.randperm(len(training_windows)).tolist()

        for window_index in window_order:
            sequence, start = training_windows[window_index]
            feature_history = sequence[start : start + window_size].to(model_device)
            target = sequence[start + window_size, :, 5:9].to(model_device)

            optimizer.zero_grad()
            prediction = model(feature_history)
            loss = nn.functional.smooth_l1_loss(prediction, target)
            loss.backward()
            optimizer.step()
            total_loss += float(loss.detach().cpu())

        epoch_losses.append(total_loss / len(training_windows))

    return epoch_losses